# 06_prott5_residue_features

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import sys
sys.path.append(r"D:\360Downloads\bioinformatics\Task\AIP")
from src.feature_prott5_residue import ProtT5ResidueExtractor, save_pickle

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "processed"
PROCESSED_DIR = DATA_DIR / "processed"
SAVE_DIR = PROCESSED_DIR / "prott5_residue"

SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train_df = pd.read_csv( INTERIM_DIR / "train.csv")
test_df = pd.read_csv( INTERIM_DIR / "test.csv")

print(train_df.shape)
print(test_df.shape)
train_df.head()

(3583, 2)
(897, 2)


,sequence,label
0,SLLLNGGCKVSNYDE,1
1,DAEFRHDSGYEVHHQ,1
2,GRTGRGKPGIYRFVAPGE,1
3,ASLKPEFVQIINAKN,1
4,KCEFQDAYVLLSEKK,1


In [ ]:
extractor = ProtT5ResidueExtractor(
    model_name="Rostlab/prot_t5_base_mt_uniref50",
    max_length=1024
)

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_base_mt_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
X_train_prott5_residue = []
for seq in tqdm(train_df["sequence"].tolist(), desc="Extract ProtT5 train residue embeddings"):
    emb = extractor.extract_one(seq)
    X_train_prott5_residue.append(emb)

X_test_prott5_residue = []
for seq in tqdm(test_df["sequence"].tolist(), desc="Extract ProtT5 test residue embeddings"):
    emb = extractor.extract_one(seq)
    X_test_prott5_residue.append(emb)

Extract ProtT5 test residue embeddings: 100%|██████████| 897/897 [00:45<00:00, 19.89it/s]


In [6]:
print("Train sample count:", len(X_train_prott5_residue))
print("Test sample count:", len(X_test_prott5_residue))
print("Example train shape:", X_train_prott5_residue[0].shape)
print("Example test shape:", X_test_prott5_residue[0].shape)

Train sample count: 3583
Test sample count: 897
Example train shape: (15, 768)
Example test shape: (15, 768)


In [7]:
save_pickle(X_train_prott5_residue, SAVE_DIR / "X_train_prott5_residue.pkl")
save_pickle(X_test_prott5_residue, SAVE_DIR / "X_test_prott5_residue.pkl")

np.save(SAVE_DIR / "y_train.npy", train_df["label"].values)
np.save(SAVE_DIR / "y_test.npy", test_df["label"].values)

train_df[["sequence", "label"]].to_csv(SAVE_DIR / "train_metadata.csv", index=False)
test_df[["sequence", "label"]].to_csv(SAVE_DIR / "test_metadata.csv", index=False)

print("ProtT5 residue-level features saved.")

ProtT5 residue-level features saved.


# esm2_residue_features

In [8]:

from src.feature_esm2_residue import ESM2ResidueExtractor, save_pickle

In [9]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SAVE_DIR = PROCESSED_DIR / "esm2_residue"

SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
lengths = train_df["sequence"].apply(len)

print("max length:", lengths.max())
print("mean length:", lengths.mean())
print("min length:", lengths.min())

max length: 30
mean length: 16.33435668434273
min length: 11


In [10]:
extractor = ESM2ResidueExtractor(
    model_name="facebook/esm2_t12_35M_UR50D",
    max_length=1024
)

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
X_train_esm2_residue = []
for seq in tqdm(train_df["sequence"].tolist(), desc="Extract ESM2 train residue embeddings"):
    emb = extractor.extract_one(seq)
    X_train_esm2_residue.append(emb)

X_test_esm2_residue = []
for seq in tqdm(test_df["sequence"].tolist(), desc="Extract ESM2 test residue embeddings"):
    emb = extractor.extract_one(seq)
    X_test_esm2_residue.append(emb)

Extract ESM2 test residue embeddings: 100%|██████████| 897/897 [00:30<00:00, 29.19it/s]


In [13]:
print("Train sample count:", len(X_train_esm2_residue))
print("Test sample count:", len(X_test_esm2_residue))
print("Example train shape:", X_train_esm2_residue[0].shape)
print("Example test shape:", X_test_esm2_residue[0].shape)

Train sample count: 3583
Test sample count: 897
Example train shape: (15, 480)
Example test shape: (15, 480)


In [14]:
save_pickle(X_train_esm2_residue, SAVE_DIR / "X_train_esm2_residue.pkl")
save_pickle(X_test_esm2_residue, SAVE_DIR / "X_test_esm2_residue.pkl")

np.save(SAVE_DIR / "y_train.npy", train_df["label"].values)
np.save(SAVE_DIR / "y_test.npy", test_df["label"].values)

train_df[["sequence", "label"]].to_csv(SAVE_DIR / "train_metadata.csv", index=False)
test_df[["sequence", "label"]].to_csv(SAVE_DIR / "test_metadata.csv", index=False)

print("ESM2 residue-level features saved.")

ESM2 residue-level features saved.
